# Phase 2 — Phân tích Transformer Cluster-SID

Notebook này load checkpoint Transformer `128 × 64 × 32` đã train và đánh giá trên validation. Mục tiêu là xác định model đang yếu ở tầng SID nào và lỗi đến từ dự đoán prefix, cluster hiếm hay độ dài session.

Các kết quả chính:

- Hit@1/5/10 và NDCG@10 cho toàn bộ SID ba tầng.
- Prefix Hit và xác suất đi tiếp đúng từ SID 0 → SID 1 → SID 2.
- Teacher-forced accuracy riêng cho từng tầng SID.
- So sánh với global popularity baseline ở cluster level.
- Breakdown theo tần suất target cluster, kích thước cluster và độ dài session.

> Đây vẫn là đánh giá cluster-SID, chưa phải item-level ranking sau candidate expansion.

## Trước khi chạy

1. Bật GPU trong Kaggle Notebook Settings.
2. Add Input là output `preprocessed` của notebook 01.
3. Add Input là output RQ-VAE có `semantic_ids.parquet`.
4. Add Input là output Transformer có `best_checkpoint.pt` hoặc `checkpoint_*.pt`.
5. Tạo Kaggle Secret `GITHUB_TOKEN` để notebook clone source hiện tại.

## 0. Cấu hình

In [ ]:
from pathlib import Path

SESSION_ROOT = None
SEMANTIC_ID_ROOT = None
TRANSFORMER_ROOT = None

GITHUB_REPOSITORY_URL = "https://github.com/nam-htran/VSF-MiniApp-Ecommerce.git"
GITHUB_BRANCH = "main"
REPOSITORY_ROOT = Path("/kaggle/working/vsf-miniapp-ecommerce-source")
OUTPUT_ROOT = Path("/kaggle/working/transformer-analysis")
AUTO_INSTALL_DEPENDENCIES = True

BATCH_SIZE = 256
MAX_EVAL_SESSIONS = None
RANDOM_SEED = 2026

print("Configuration loaded.")

## 1. Cài dependency và clone source

In [ ]:
import importlib.metadata as metadata
import os
import subprocess
import sys

from packaging.version import Version

requirements = {
    "accelerate": "1.0.0",
    "einops": "0.8.0",
    "transformers": "4.46.0",
    "pyarrow": "16.0.0",
    "matplotlib": "3.8.0",
}
packages_to_install = []
for distribution, minimum_version in requirements.items():
    try:
        installed_version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed_version = None
    if installed_version is None or Version(installed_version) < Version(minimum_version):
        packages_to_install.append(f"{distribution}>={minimum_version}")

if AUTO_INSTALL_DEPENDENCIES and packages_to_install:
    print("Installing:", packages_to_install)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages_to_install])

import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running the analysis.")

from kaggle_secrets import UserSecretsClient

github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
git_environment = {**os.environ, "GITHUB_TOKEN": github_token, "GIT_TERMINAL_PROMPT": "0"}
credential_helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
git = ["git", "-c", f"credential.helper={credential_helper}"]

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run([*git, "-C", str(REPOSITORY_ROOT), "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True, env=git_environment)
elif REPOSITORY_ROOT.exists():
    raise FileExistsError(f"Clone target is not a Git repository: {REPOSITORY_ROOT}")
else:
    subprocess.run([*git, "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPOSITORY_URL, str(REPOSITORY_ROOT)], check=True, env=git_environment)
del github_token, git_environment

SOURCE_ROOT = REPOSITORY_ROOT / "ai-recommendation/src"
if not (SOURCE_ROOT / "modules/model.py").is_file():
    raise FileNotFoundError(f"Transformer source not found: {SOURCE_ROOT}")
sys.path.insert(0, str(SOURCE_ROOT))

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("SOURCE_ROOT:", SOURCE_ROOT)

## 2. Tìm session, Semantic ID và checkpoint

In [ ]:
def is_session_root(path):
    path = Path(path)
    return (path / "model_sessions_train.parquet").is_file() and (path / "model_sessions_validation.parquet").is_file()


def is_semantic_id_root(path):
    return (Path(path) / "semantic_ids.parquet").is_file()


def checkpoint_paths(path):
    path = Path(path)
    best = path / "best_checkpoint.pt"
    if best.is_file():
        return [best]
    return sorted(path.glob("checkpoint_*.pt"), key=lambda item: int(item.stem.rsplit("_", 1)[1]))


def locate_root(explicit, validator, local_candidates, glob_pattern, description):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if validator(root):
            return root
        raise FileNotFoundError(f"{description} was not found in {root}")

    for candidate in local_candidates:
        if validator(candidate):
            return Path(candidate).resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for match in kaggle_input.glob(glob_pattern):
            if validator(match.parent):
                return match.parent.resolve()
    raise FileNotFoundError(f"{description} was not found. Add the corresponding Kaggle Dataset or set its root explicitly.")


cwd = Path.cwd().resolve()
SESSION_ROOT = locate_root(
    SESSION_ROOT,
    is_session_root,
    [Path("/kaggle/working/preprocessed"), cwd / "preprocessed", cwd.parent / "preprocessed"],
    "**/model_sessions_validation.parquet",
    "Notebook 01 session artifacts",
)
SEMANTIC_ID_ROOT = locate_root(
    SEMANTIC_ID_ROOT,
    is_semantic_id_root,
    [Path("/kaggle/working/rq-vae"), cwd / "output/rq-vae", cwd / "ai-recommendation/output/rq-vae"],
    "**/semantic_ids.parquet",
    "Notebook 03 Semantic ID artifacts",
)
transformer_candidates = [Path("/kaggle/working/transformer"), cwd / "transformer", cwd.parent / "transformer"]
if TRANSFORMER_ROOT is not None:
    transformer_candidates.insert(0, Path(TRANSFORMER_ROOT).expanduser().resolve())
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    transformer_candidates.extend(path.parent for path in kaggle_input.glob("**/best_checkpoint.pt"))
    transformer_candidates.extend(path.parent for path in kaggle_input.glob("**/checkpoint_*.pt"))
TRANSFORMER_ROOT = next((path.resolve() for path in transformer_candidates if checkpoint_paths(path)), None)
if TRANSFORMER_ROOT is None:
    raise FileNotFoundError("Notebook 04 Transformer checkpoint was not found. Add its Kaggle Dataset or set TRANSFORMER_ROOT.")
CHECKPOINT_PATH = checkpoint_paths(TRANSFORMER_ROOT)[-1]
SEMANTIC_IDS_PATH = SEMANTIC_ID_ROOT / "semantic_ids.parquet"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("SESSION_ROOT:", SESSION_ROOT)
print("SEMANTIC_IDS_PATH:", SEMANTIC_IDS_PATH)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

## 3. Load model và validation data

In [ ]:
from torch.utils.data import DataLoader, Subset

from data.processed import RecDataset, SeqData
from data.utils import batch_to
from modules.model import EncoderDecoderRetrievalModel
from modules.tokenizer.semids import PrecomputedSemanticIdTokenizer

CODEBOOK_SIZES = [128, 64, 32]
NUM_HIERARCHIES = len(CODEBOOK_SIZES)
TOP_K = 10
device = torch.device("cuda")

semantic_ids = pd.read_parquet(SEMANTIC_IDS_PATH)
sid_columns = [f"sid_{index}" for index in range(NUM_HIERARCHIES)]
expected_columns = ["product_index", "product_id", *sid_columns]
if list(semantic_ids.columns) != expected_columns:
    raise ValueError(f"Unexpected Semantic ID columns: {list(semantic_ids.columns)}")
if not np.array_equal(semantic_ids["product_index"].to_numpy(), np.arange(len(semantic_ids))):
    raise ValueError("product_index must match the Semantic ID row order")

codebooks = torch.from_numpy(semantic_ids[sid_columns].to_numpy(dtype=np.int16, copy=True))
model = EncoderDecoderRetrievalModel(
    codebooks=codebooks,
    codebook_sizes=CODEBOOK_SIZES,
    t5_d_model=384,
    t5_num_heads=6,
    t5_d_ff=1024,
    t5_num_layers=4,
    top_k_for_generation=TOP_K,
    should_add_sep_token=True,
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
checkpoint_step = int(checkpoint["iter"]) + 1 if "iter" in checkpoint else None

def checkpoint_key(key):
    prefixes = ("_orig_mod.", "module.")
    changed = True
    while changed:
        changed = False
        for prefix in prefixes:
            if key.startswith(prefix):
                key = key[len(prefix):]
                changed = True
    return key

state_dict = {checkpoint_key(key): value for key, value in checkpoint["model"].items()}
model.load_state_dict(state_dict, strict=True)
model.to(device).eval()
tokenizer = PrecomputedSemanticIdTokenizer(codebooks).to(device)

validation_dataset = SeqData(
    root=SESSION_ROOT,
    index_path=SEMANTIC_IDS_PATH,
    session_root=SESSION_ROOT,
    dataset=RecDataset.VMARKET,
    is_train=False,
)
selected_indices = np.arange(len(validation_dataset))
if MAX_EVAL_SESSIONS is not None and MAX_EVAL_SESSIONS < len(selected_indices):
    selected_indices = np.sort(np.random.default_rng(RANDOM_SEED).choice(selected_indices, MAX_EVAL_SESSIONS, replace=False))
evaluation_dataset = Subset(validation_dataset, selected_indices.tolist())
evaluation_loader = DataLoader(evaluation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Checkpoint step:", checkpoint_step if checkpoint_step is not None else "unknown")
print("Validation sessions:", f"{len(validation_dataset):,}")
print("Sessions selected:", f"{len(evaluation_dataset):,}")

## 4. Tính popularity và đặc tính target cluster

Popularity được tính hoàn toàn global từ target của train sessions, không lọc locale.

In [ ]:
sid_values = semantic_ids[sid_columns].to_numpy(dtype=np.int64)
product_cluster_keys = sid_values[:, 0].copy()
for layer in range(1, NUM_HIERARCHIES):
    product_cluster_keys = product_cluster_keys * CODEBOOK_SIZES[layer] + sid_values[:, layer]
product_to_cluster = pd.Series(product_cluster_keys, index=semantic_ids["product_id"].astype(str))

train_targets = pd.read_parquet(SESSION_ROOT / "model_sessions_train.parquet", columns=["next_item"])["next_item"].astype(str)
train_target_keys = train_targets.map(product_to_cluster)
if train_target_keys.isna().any():
    raise ValueError(f"Unknown train targets: {int(train_target_keys.isna().sum()):,}")
cluster_frequency = train_target_keys.astype(np.int64).value_counts()
cluster_size = pd.Series(product_cluster_keys).value_counts()
popular_cluster_keys = cluster_frequency.index.to_numpy(dtype=np.int64)[:TOP_K]

print("Train targets:", f"{len(train_targets):,}")
print("Target clusters observed in train:", f"{len(cluster_frequency):,}")
print("Most frequent cluster count:", f"{int(cluster_frequency.iloc[0]):,}")
del train_targets, train_target_keys

## 5. Chạy diagnostic evaluation

Teacher forcing cho model nhìn thấy prefix SID thật. Nếu teacher-forced accuracy cao nhưng autoregressive prefix thấp, lỗi chính là propagation/search. Nếu cả hai cùng thấp, tầng dự đoán đó chưa học đủ.

In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm

columns = defaultdict(list)

with torch.inference_mode():
    for batch in tqdm(evaluation_loader, desc="Analyzing validation"):
        batch = batch_to(batch, device)
        tokenized = tokenizer(batch)
        actual = tokenized.sem_ids_fut[:, :NUM_HIERARCHIES]

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            generated = model.generate_next_sem_id(tokenized).sem_ids
            encoder_output, encoder_mask = model.encoder_forward_pass(tokenized.seq_mask.long(), tokenized.sem_ids)
            decoder_output = model.decoder_forward_pass(
                future_ids=actual,
                encoder_output=encoder_output,
                attention_mask_for_encoder=encoder_mask,
                use_cache=False,
            )[:, :-1]

        exact_match = generated.eq(actual.unsqueeze(1)).all(dim=-1)
        has_match = exact_match.any(dim=1)
        exact_rank = torch.where(has_match, exact_match.float().argmax(dim=1) + 1, 0)
        ndcg = torch.zeros_like(exact_rank, dtype=torch.float32)
        ndcg[has_match] = 1.0 / torch.log2(exact_rank[has_match].float() + 1.0)

        columns["rank"].append(exact_rank.cpu().numpy())
        columns["ndcg"].append(ndcg.cpu().numpy())
        columns["session_length"].append(batch.seq_mask.sum(dim=1).cpu().numpy())

        target_indices = batch.ids_fut[:, 0].cpu().numpy()
        target_keys = product_cluster_keys[target_indices]
        columns["target_cluster_key"].append(target_keys)
        columns["target_frequency"].append(np.array([cluster_frequency.get(key, 0) for key in target_keys]))
        columns["cluster_size"].append(np.array([cluster_size.get(key, 0) for key in target_keys]))

        for k in (1, 5, 10):
            columns[f"hit_{k}"].append(exact_match[:, :k].any(dim=1).cpu().numpy())

        for depth in range(1, NUM_HIERARCHIES + 1):
            prefix_match = generated[:, :, :depth].eq(actual[:, None, :depth]).all(dim=-1)
            columns[f"top1_prefix_{depth}"].append(prefix_match[:, 0].cpu().numpy())
            columns[f"prefix_{depth}_hit_10"].append(prefix_match.any(dim=1).cpu().numpy())

        for layer in range(NUM_HIERARCHIES):
            logits = model.decoder_mlp[layer](decoder_output[:, layer]).float()
            predicted_codes = logits.topk(TOP_K, dim=-1).indices
            layer_match = predicted_codes.eq(actual[:, layer].unsqueeze(1))
            for k in (1, 5, 10):
                columns[f"tf_sid_{layer}_hit_{k}"].append(layer_match[:, :k].any(dim=1).cpu().numpy())

results = pd.DataFrame({name: np.concatenate(values) for name, values in columns.items()})
results["frequency_bucket"] = pd.cut(
    results["target_frequency"],
    bins=[-1, 0, 10, 100, 1000, np.inf],
    labels=["unseen", "1-10", "11-100", "101-1000", ">1000"],
)
results["cluster_size_bucket"] = pd.cut(
    results["cluster_size"],
    bins=[0, 1, 5, 20, np.inf],
    labels=["1", "2-5", "6-20", ">20"],
)
results["session_length_bucket"] = pd.cut(
    results["session_length"],
    bins=[0, 1, 3, 5, 10, 20],
    labels=["1", "2-3", "4-5", "6-10", "11-20"],
)

print("Analyzed sessions:", f"{len(results):,}")
display(results.head())

## 6. Tổng hợp kết quả

In [ ]:
overall = pd.DataFrame(
    {
        "model": ["Transformer"],
        "H@1": [results["hit_1"].mean()],
        "H@5": [results["hit_5"].mean()],
        "H@10": [results["hit_10"].mean()],
        "NDCG@10": [results["ndcg"].mean()],
    }
)

target_keys = results["target_cluster_key"].to_numpy()
popularity_match = target_keys[:, None] == popular_cluster_keys[None, :]
popularity_found = popularity_match.any(axis=1)
popularity_rank = np.where(popularity_found, popularity_match.argmax(axis=1) + 1, 0)
popularity_ndcg = np.zeros(len(popularity_rank), dtype=np.float64)
popularity_ndcg[popularity_found] = 1.0 / np.log2(popularity_rank[popularity_found] + 1.0)
popularity = pd.DataFrame(
    {
        "model": ["Global popularity"],
        "H@1": [(popularity_rank == 1).mean()],
        "H@5": [((popularity_rank > 0) & (popularity_rank <= 5)).mean()],
        "H@10": [(popularity_rank > 0).mean()],
        "NDCG@10": [popularity_ndcg.mean()],
    }
)
comparison = pd.concat([overall, popularity], ignore_index=True)
display(comparison.style.format({column: "{:.4f}" for column in ["H@1", "H@5", "H@10", "NDCG@10"]}))

prefix_summary = pd.DataFrame(
    {
        "depth": [1, 2, 3],
        "top1_prefix_accuracy": [results[f"top1_prefix_{depth}"].mean() for depth in range(1, 4)],
        "prefix_hit_10": [results[f"prefix_{depth}_hit_10"].mean() for depth in range(1, 4)],
    }
)
prefix_summary["top1_conditional_continuation"] = [
    prefix_summary.loc[0, "top1_prefix_accuracy"],
    prefix_summary.loc[1, "top1_prefix_accuracy"] / max(prefix_summary.loc[0, "top1_prefix_accuracy"], 1e-12),
    prefix_summary.loc[2, "top1_prefix_accuracy"] / max(prefix_summary.loc[1, "top1_prefix_accuracy"], 1e-12),
]
display(prefix_summary.style.format({column: "{:.4f}" for column in prefix_summary.columns if column != "depth"}))

teacher_forced = pd.DataFrame(
    [
        {
            "SID layer": layer,
            **{f"TF H@{k}": results[f"tf_sid_{layer}_hit_{k}"].mean() for k in (1, 5, 10)},
        }
        for layer in range(NUM_HIERARCHIES)
    ]
)
display(teacher_forced.style.format({column: "{:.4f}" for column in teacher_forced.columns if column != "SID layer"}))

In [ ]:
def grouped_metrics(frame, column):
    return (
        frame.groupby(column, observed=True)
        .agg(
            sessions=("rank", "size"),
            H_1=("hit_1", "mean"),
            H_5=("hit_5", "mean"),
            H_10=("hit_10", "mean"),
            NDCG_10=("ndcg", "mean"),
            prefix_1_H_10=("prefix_1_hit_10", "mean"),
            prefix_2_H_10=("prefix_2_hit_10", "mean"),
        )
        .reset_index()
    )

frequency_summary = grouped_metrics(results, "frequency_bucket")
cluster_size_summary = grouped_metrics(results, "cluster_size_bucket")
session_length_summary = grouped_metrics(results, "session_length_bucket")

print("By target-cluster frequency")
display(frequency_summary.style.format({"H_1": "{:.4f}", "H_5": "{:.4f}", "H_10": "{:.4f}", "NDCG_10": "{:.4f}", "prefix_1_H_10": "{:.4f}", "prefix_2_H_10": "{:.4f}"}))
print("By number of products in the target cluster")
display(cluster_size_summary.style.format({"H_1": "{:.4f}", "H_5": "{:.4f}", "H_10": "{:.4f}", "NDCG_10": "{:.4f}", "prefix_1_H_10": "{:.4f}", "prefix_2_H_10": "{:.4f}"}))
print("By session length")
display(session_length_summary.style.format({"H_1": "{:.4f}", "H_5": "{:.4f}", "H_10": "{:.4f}", "NDCG_10": "{:.4f}", "prefix_1_H_10": "{:.4f}", "prefix_2_H_10": "{:.4f}"}))

## 7. Biểu đồ và lưu artifact

In [ ]:
import json
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
teacher_forced.set_index("SID layer")[["TF H@1", "TF H@5", "TF H@10"]].plot.bar(ax=axes[0], ylim=(0, 1), title="Teacher-forced accuracy")
frequency_summary.plot.bar(x="frequency_bucket", y="H_10", ax=axes[1], ylim=(0, 1), legend=False, title="H@10 by target frequency")
session_length_summary.plot.bar(x="session_length_bucket", y="H_10", ax=axes[2], ylim=(0, 1), legend=False, title="H@10 by session length")
plt.tight_layout()
plt.show()

results.to_parquet(OUTPUT_ROOT / "transformer_analysis_per_session.parquet", index=False)
comparison.to_csv(OUTPUT_ROOT / "model_vs_popularity.csv", index=False)
prefix_summary.to_csv(OUTPUT_ROOT / "prefix_summary.csv", index=False)
teacher_forced.to_csv(OUTPUT_ROOT / "teacher_forced_summary.csv", index=False)
frequency_summary.to_csv(OUTPUT_ROOT / "frequency_summary.csv", index=False)
cluster_size_summary.to_csv(OUTPUT_ROOT / "cluster_size_summary.csv", index=False)
session_length_summary.to_csv(OUTPUT_ROOT / "session_length_summary.csv", index=False)

summary = {
    "checkpoint": str(CHECKPOINT_PATH),
    "checkpoint_step": checkpoint_step,
    "sessions": len(results),
    "h_at_1": float(results["hit_1"].mean()),
    "h_at_5": float(results["hit_5"].mean()),
    "h_at_10": float(results["hit_10"].mean()),
    "ndcg_at_10": float(results["ndcg"].mean()),
}
(OUTPUT_ROOT / "transformer_analysis_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Analysis artifacts saved to:", OUTPUT_ROOT)

## Cách đọc kết quả

- Teacher-forced tốt nhưng prefix autoregressive giảm mạnh: lỗi tích lũy qua các tầng hoặc beam search là nút thắt.
- Teacher-forced của SID 1/2 cũng thấp: model hoặc representation chưa học đủ các tầng sâu.
- Cluster hiếm kém rõ rệt: cân nhắc sampling hoặc loss weighting theo frequency.
- Cluster lớn kém rõ rệt: cluster-SID chưa đủ để chọn item; candidate ranking sẽ quan trọng.
- Session dài tốt hơn session ngắn: tín hiệu hành vi đang hữu ích. Session dài kém hơn: kiểm tra truncation và cách model biểu diễn lịch sử.
- Chỉ nên tăng `d_ff` hoặc số layer khi train và validation đều cho thấy underfitting, không quyết định chỉ từ total loss.